In [3]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [4]:
def parse_solinst_csv(filepath):
    """
    Parse Solinst levellogger CSV export.
    
    The Solinst format has metadata in the first ~13 rows, then data starting
    with headers on row 14. The LEVEL column represents water column height 
    above the sensor in meters.
    
    Parameters:
    -----------
    filepath : str or Path
        Path to the compensated levellogger CSV file
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with datetime, water_column_m, and temperature_C columns
    """
    # Read the data, skipping metadata rows (typically first 14 rows)
    # Solinst format has headers on row 14 (0-indexed row 13)
    # Use latin-1 encoding to handle degree symbol (°C)
    df = pd.read_csv(filepath, skiprows=13, encoding='latin-1')
    
    # Extract logger name from file location info (row 6)
    with open(filepath, 'r', encoding='latin-1') as f:
        lines = f.readlines()
        location_line = lines[5].strip()  # Row 6 (0-indexed 5)
        logger_name = location_line.split(',')[0]
    
    # Create datetime column
    df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
    
    # Rename columns for clarity
    df = df.rename(columns={
        'LEVEL': 'water_column_m',
        'TEMPERATURE': 'temperature_C'
    })
    
    # Select relevant columns
    df = df[['datetime', 'water_column_m', 'temperature_C']].copy()
    
    # Remove any rows with missing data
    df = df.dropna()
    
    return df, logger_name

In [5]:
def convert_to_water_level_below_ground(df, sensor_depth_cm):
    """
    Convert water column height to water level below ground surface.
    
    Formula:
    Water Level Below Ground (cm) = Sensor Depth Below Ground (cm) - Water Column Height (cm)
    
    Where:
    - Sensor Depth Below Ground is negative (e.g., -211.1 cm means 211.1 cm below surface)
    - Water Column Height is converted from meters to cm
    - Result is negative when water table is below ground (typical)
    - Result is positive when water is above ground surface (rare, flooding/artesian)
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with water_column_m column
    sensor_depth_cm : float
        Depth of sensor below ground in cm (negative value)
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with added water_level_below_ground_cm column
    """
    df = df.copy()
    
    # Convert water column from meters to cm
    water_column_cm = df['water_column_m'] * 100
    
    # Calculate water level below ground
    # sensor_depth_cm is negative (e.g., -211.1)
    # water_column_cm is positive (e.g., 196.1)
    # Result: -211.1 - 196.1 = -407.2 cm (water table is 407.2 cm below surface)
    # But we want: -211.1 + 196.1 = -15.0 cm if sensor at -211.1 and water column is 196.1 cm high
    # Wait, let me reconsider...
    
    # If sensor is at -211.1 cm (211.1 cm below ground)
    # And water column above sensor is 196.1 cm
    # Then water level is at: -211.1 + 196.1 = -15.0 cm (15 cm below ground)
    
    # Correction: 
    df['water_level_below_ground_cm'] = sensor_depth_cm + water_column_cm
    
    return df

In [6]:
def process_single_logger(filepath, sensor_depths_df, output_dir):
    """
    Process a single levellogger file.
    
    Parameters:
    -----------
    filepath : Path
        Path to compensated CSV file
    sensor_depths_df : pd.DataFrame
        Reference dataframe with logger_ID and sensor_below_ground columns
    output_dir : Path
        Directory to save output files
        
    Returns:
    --------
    pd.DataFrame or None
        Processed dataframe, or None if logger not in reference file
    """
    print(f"Processing: {filepath.name}")
    
    # Parse the levellogger data
    df, logger_name = parse_solinst_csv(filepath)
    
    # Look up sensor depth
    sensor_match = sensor_depths_df[sensor_depths_df['logger_ID'] == logger_name]
    
    if len(sensor_match) == 0:
        print(f"  WARNING: {logger_name} not found in levellogger_below_ground.csv - SKIPPING")
        return None
    
    sensor_depth_cm = sensor_match['sensor_below_ground'].values[0]
    print(f"  Logger: {logger_name}, Sensor depth: {sensor_depth_cm} cm below ground")
    
    # Convert to water level below ground
    df = convert_to_water_level_below_ground(df, sensor_depth_cm)
    
    # Add logger name column
    df.insert(0, 'logger_ID', logger_name)
    
    # Calculate summary statistics
    print(f"  Date range: {df['datetime'].min()} to {df['datetime'].max()}")
    print(f"  Number of readings: {len(df)}")
    print(f"  Water level below ground (cm):")
    print(f"    Mean: {df['water_level_below_ground_cm'].mean():.2f}")
    print(f"    Min: {df['water_level_below_ground_cm'].min():.2f}")
    print(f"    Max: {df['water_level_below_ground_cm'].max():.2f}")
    
    # Save output file
    output_filename = filepath.stem.replace('Compensated', 'WaterLevelBG') + '.csv'
    output_path = output_dir / output_filename
    
    df.to_csv(output_path, index=False)
    print(f"  Saved to: {output_path}")
    print()
    
    return df

In [7]:
def process_all_loggers(data_dir, output_dir=None, skip_sites=None):
    """
    Process all compensated levellogger files in a directory.
    
    Parameters:
    -----------
    data_dir : str or Path
        Directory containing levellogger data files
    output_dir : str or Path, optional
        Directory to save output files. If None, creates 'processed' subdirectory
    skip_sites : list, optional
        List of site names to skip (e.g., ['GCB_piezo', 'GCB_well'])
        
    Returns:
    --------
    dict
        Dictionary mapping logger names to processed DataFrames
    """
    data_dir = Path(data_dir)
    
    if skip_sites is None:
        skip_sites = []
    
    # Set up output directory
    if output_dir is None:
        output_dir = data_dir / 'processed'
    else:
        output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True)
    
    print(f"Processing levellogger data from: {data_dir}")
    print(f"Output directory: {output_dir}")
    if skip_sites:
        print(f"Skipping sites: {', '.join(skip_sites)}")
    print()
    
    # Load sensor depth reference file
    sensor_depths_file = data_dir / 'levellogger_below_ground.csv'
    if not sensor_depths_file.exists():
        raise FileNotFoundError(
            f"Reference file not found: {sensor_depths_file}\n"
            f"This file must contain logger_ID and sensor_below_ground columns"
        )
    
    sensor_depths_df = pd.read_csv(sensor_depths_file)
    print(f"Loaded sensor depths for {len(sensor_depths_df)} loggers\n")
    print("=" * 80)
    
    # Find all compensated CSV files
    compensated_files = list(data_dir.glob('*Compensated.csv'))
    print(f"\nFound {len(compensated_files)} compensated files to process\n")
    
    # Process each file
    results = {}
    for filepath in sorted(compensated_files):
        # Check if this site should be skipped
        should_skip = False
        for skip_site in skip_sites:
            if skip_site in filepath.stem:
                print(f"Skipping: {filepath.name} (manually compensated)")
                should_skip = True
                break
        
        if should_skip:
            continue
        
        df = process_single_logger(filepath, sensor_depths_df, output_dir)
        if df is not None:
            logger_name = df['logger_ID'].iloc[0]
            results[logger_name] = df
    
    print("=" * 80)
    print(f"Processing complete! Processed {len(results)} files successfully.")
    print(f"Output files saved to: {output_dir}")
    
    return results

In [8]:
def create_combined_output(results, output_dir):
    """
    Create a combined file with all loggers for easier analysis.
    
    Parameters:
    -----------
    results : dict
        Dictionary of processed DataFrames from process_all_loggers()
    output_dir : Path
        Directory to save combined file
    """
    if not results:
        print("No data to combine.")
        return
    
    # Combine all dataframes
    combined = pd.concat(results.values(), ignore_index=True)
    
    # Sort by logger and datetime
    combined = combined.sort_values(['logger_ID', 'datetime'])
    
    # Save combined file
    output_path = output_dir / 'all_loggers_water_level_below_ground.csv'
    combined.to_csv(output_path, index=False)
    
    print(f"\nCombined file saved to: {output_path}")
    print(f"Total records: {len(combined)}")
    print(f"Loggers included: {combined['logger_ID'].nunique()}")
    
    return combined


In [9]:
def calculate_wtp_statistics(results, output_dir, 
                             start_date='2025-08-01 00:00:00', 
                             end_date='2025-09-17 23:45:00'):
    """
    Calculate Water Table Position (WTP) statistics for each logger
    within a specified date range.
    
    Parameters:
    -----------
    results : dict
        Dictionary of processed DataFrames from process_all_loggers()
    output_dir : Path
        Directory to save statistics file
    start_date : str
        Start of date range (format: 'YYYY-MM-DD HH:MM:SS')
    end_date : str
        End of date range (format: 'YYYY-MM-DD HH:MM:SS')
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with WTP statistics for each logger
    """
    if not results:
        print("No data to calculate WTP statistics.")
        return None
    
    output_dir = Path(output_dir)
    
    print("\n" + "=" * 80)
    print(f"CALCULATING WTP STATISTICS")
    print("=" * 80)
    print(f"Date range: {start_date} to {end_date}")
    print()
    
    # Convert date strings to datetime
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)
    
    # Calculate statistics for each logger
    stats_list = []
    
    for logger_name, df in results.items():
        # Ensure datetime column is datetime type
        df['datetime'] = pd.to_datetime(df['datetime'])
        
        # Filter to date range
        mask = (df['datetime'] >= start_dt) & (df['datetime'] <= end_dt)
        df_period = df[mask].copy()
        
        if len(df_period) == 0:
            print(f"  {logger_name}: No data in specified date range - SKIPPING")
            continue
        
        # Calculate statistics
        stats = {
            'logger_ID': logger_name,
            'n_records': len(df_period),
            'start_date': df_period['datetime'].min(),
            'end_date': df_period['datetime'].max(),
            'WTP_mean': df_period['water_level_below_ground_cm'].mean(),
            'WTP_median': df_period['water_level_below_ground_cm'].median(),
            'WTP_stdev': df_period['water_level_below_ground_cm'].std(),
            'WTP_min': df_period['water_level_below_ground_cm'].min(),
            'WTP_max': df_period['water_level_below_ground_cm'].max(),
            'WTP_range': (df_period['water_level_below_ground_cm'].max() - 
                         df_period['water_level_below_ground_cm'].min())
        }
        
        stats_list.append(stats)
        
        print(f"  {logger_name}: {len(df_period)} records, "
              f"Mean WTP = {stats['WTP_mean']:.2f} cm")
    
    # Create DataFrame
    wtp_stats = pd.DataFrame(stats_list)
    
    # Round values
    wtp_stats = wtp_stats.round({
        'WTP_mean': 2,
        'WTP_median': 2,
        'WTP_stdev': 2,
        'WTP_min': 2,
        'WTP_max': 2,
        'WTP_range': 2
    })
    
    # Sort by logger name
    wtp_stats = wtp_stats.sort_values('logger_ID')
    
    # Save to CSV
    output_file = output_dir / 'WTP_statistics_Aug1_Sept17.csv'
    wtp_stats.to_csv(output_file, index=False)
    
    print()
    print("=" * 80)
    print(f"WTP statistics saved to: {output_file.name}")
    print(f"Total loggers: {len(wtp_stats)}")
    print()
    
    # Print summary
    print("WTP Statistics Summary:")
    print(wtp_stats[['logger_ID', 'n_records', 'WTP_mean', 'WTP_median', 
                     'WTP_stdev', 'WTP_min', 'WTP_max']].to_string(index=False))
    
    return wtp_stats


In [13]:
# Example usage
if __name__ == "__main__":
    # Set your data directory path
    data_dir = r"C:\Users\leila\Dropbox\MayoWetlands\LevelloggerData\2025"
    
    # Process all files (skip GCB sites - manually compensated)
    skip_gcb = ['GCB_piezo', 'GCB_well']
    results = process_all_loggers(data_dir, skip_sites=skip_gcb)
    
    # Create combined output file
    output_dir = Path(data_dir) / 'processed'
    combined_df = create_combined_output(results, output_dir)
    
    # Calculate WTP statistics for Aug 1 - Sept 12 period
    wtp_stats = calculate_wtp_statistics(
        results, 
        output_dir,
        start_date='2025-08-01 00:00:00',
        end_date='2025-09-12 23:45:00'
    )
    
    print("\nProcessing complete!")

Processing levellogger data from: C:\Users\leila\Dropbox\MayoWetlands\LevelloggerData\2025
Output directory: C:\Users\leila\Dropbox\MayoWetlands\LevelloggerData\2025\processed
Skipping sites: GCB_piezo, GCB_well

Loaded sensor depths for 27 loggers


Found 28 compensated files to process

Processing: APA_piezoCompensated.csv
  Logger: APA_piezo, Sensor depth: -225.1 cm below ground
  Date range: 2025-06-11 14:00:00 to 2025-09-19 19:15:00
  Number of readings: 9622
  Water level below ground (cm):
    Mean: -13.87
    Min: -230.00
    Max: -6.50
  Saved to: C:\Users\leila\Dropbox\MayoWetlands\LevelloggerData\2025\processed\APA_piezoWaterLevelBG.csv

Processing: APA_wellCompensated.csv
  Logger: APA_well, Sensor depth: -69.9 cm below ground
  Date range: 2025-06-11 14:00:00 to 2025-09-19 18:30:00
  Number of readings: 9619
  Water level below ground (cm):
    Mean: -5.16
    Min: -76.10
    Max: -2.60
  Saved to: C:\Users\leila\Dropbox\MayoWetlands\LevelloggerData\2025\processed\APA_well